In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import logging
from pathlib import Path
import os

logging.getLogger('WDM').setLevel(logging.NOTSET)

In [ ]:
def scrape_stock(url: str) -> pd.DataFrame:
    #Scrapes historical stock data from a Yahoo Finance URL."""
    print(f"Starting to scrape: {url}")
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument('log-level=3')
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)

    data = []
    try:
        driver.get(url)
        time.sleep(5)
        rows = driver.find_elements(By.XPATH, '//table[contains(@class,"table")]/tbody/tr')
        
        for row in rows:
            cols = row.find_elements(By.TAG_NAME, "td")
            if len(cols) >= 7:
                date = cols[0].text
                if "Dividend" not in date and "Stock Split" not in date:
                    data.append([
                        cols[0].text, cols[1].text, cols[2].text,
                        cols[3].text, cols[4].text, cols[6].text
                    ])
    except Exception as e:
        print(f"An error occurred while scraping {url}: {e}")
    finally:
        driver.quit()

    return pd.DataFrame(data, columns=["Date", "Open", "High", "Low", "Close", "Volume"])

In [ ]:
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    # Converting columns to correct data types.
    df_clean = df.copy()
    
    df_clean = df_clean[df_clean['Volume'] != '0']
    df_clean.dropna(inplace=True)

    if df_clean.empty:
        print("Data is empty after initial cleaning, skipping further processing.")
        return df_clean

    df_clean["Date"] = pd.to_datetime(df_clean["Date"], format="%b %d, %Y")
    
    for col in ["Open", "High", "Low", "Close", "Volume"]:
        df_clean[col] = df_clean[col].str.replace(",", "", regex=True).astype(float)

    df_clean["Volume"] = df_clean["Volume"].astype("int64")

    print("Data cleaning complete.")
    return df_clean

In [ ]:
if __name__ == "__main__":
    target_urls = [
        "https://finance.yahoo.com/quote/TLKM.JK/history/?period1=1599045267&period2=1756811655",
        "https://finance.yahoo.com/quote/ASII.JK/history/?period1=1599045376&period2=1756811772",
        "https://finance.yahoo.com/quote/ANTM.JK/history/?period1=1599045961&period2=1756812357",
        "https://finance.yahoo.com/quote/ADRO.JK/history/?period1=1599046061&period2=1756812456",
        "https://finance.yahoo.com/quote/BRIS.JK/history/?period1=1599046226&period2=1756812622",
        "https://finance.yahoo.com/quote/BBCA.JK/history/?period1=1599046266&period2=1756812663",
        "https://finance.yahoo.com/quote/BMRI.JK/history/?period1=1599046309&period2=1756812705",
        "https://finance.yahoo.com/quote/BBNI.JK/history/?period1=1599046342&period2=1756812738",
        "https://finance.yahoo.com/quote/UNTR.JK/history/?period1=1599046486&period2=1756812883",
        "https://finance.yahoo.com/quote/PGAS.JK/history/?period1=1599046584&period2=1756812981"
    ]
    
    save_directory = Path("data/raw")
    
    save_directory.mkdir(parents=True, exist_ok=True)

    tickers = [
        "TLKM", "ASII", "ANTM", "ADRO", "BRIS", 
        "BBCA", "BMRI", "BBNI", "UNTR", "PGAS"
    ]
    
    for url, ticker in zip(target_urls, tickers):
        df_raw = scrape_stock(url)
        
        if not df_raw.empty:
            df_clean = clean_dataframe(df_raw)
            
            if not df_clean.empty:
                filename = f"{ticker}_history.csv"
                full_path = save_directory / filename
                df_clean.to_csv(full_path, index=False)
                print(f"Data successfully saved to: {full_path}")        
        print("-" * 40)

In [ ]:
# Split historical CSV files into Kafka sample (for stream data) and remaining data (for batch data)
for file in os.listdir(save_directory):
    if file.endswith("_history.csv"):
        src_path = save_directory / file
        df = pd.read_csv(src_path)
        
        # Take first 10 rows
        df_kafka = df.head(10)
        
        # Save to new file with _kafka.csv suffix
        kafka_filename = file.replace("_history.csv", "_kafka.csv")
        kafka_path = save_directory / kafka_filename
        df_kafka.to_csv(kafka_path, index=False)
        
        # Remove first 10 rows from original dataframe and overwrite source file
        df_rest = df.iloc[10:]
        df_rest.to_csv(src_path, index=False)